# Prototyping Functions to Ingest Data

In [2]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from typing import List
from fredapi import Fred

## EIA Spot Prices

In [3]:
load_dotenv()
EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data
frequency = "weekly" # Frequency of the spot prices, either daily, weekly, or monthly.
series_ids = ["RBRTE", "RWTC"] # ID's of series we're pulling (Brent, WTI)
length = 5000

def series_line(series_ids) -> List[str]: # Build the text used to specify what series we want
    text = [f"&facets[series][]={id}" for id in series_ids]
    return ''.join(text)


URL_BASE = f"https://api.eia.gov/v2/petroleum/pri/spt/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}"
response = requests.get(url= URL_BASE) # Make request to EIA API
data = response.json()

In [4]:
df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
df.head()

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
4146,1986-01-03,YCUOK,NA,EPCWTI,WTI Crude Oil,PF4,Spot Price FOB,RWTC,"Cushing, OK WTI Spot Price FOB (Dollars per Ba...",25.78,$/BBL
4145,1986-01-10,YCUOK,NA,EPCWTI,WTI Crude Oil,PF4,Spot Price FOB,RWTC,"Cushing, OK WTI Spot Price FOB (Dollars per Ba...",25.99,$/BBL
4144,1986-01-17,YCUOK,NA,EPCWTI,WTI Crude Oil,PF4,Spot Price FOB,RWTC,"Cushing, OK WTI Spot Price FOB (Dollars per Ba...",24.57,$/BBL
4143,1986-01-24,YCUOK,NA,EPCWTI,WTI Crude Oil,PF4,Spot Price FOB,RWTC,"Cushing, OK WTI Spot Price FOB (Dollars per Ba...",20.31,$/BBL
4142,1986-01-31,YCUOK,NA,EPCWTI,WTI Crude Oil,PF4,Spot Price FOB,RWTC,"Cushing, OK WTI Spot Price FOB (Dollars per Ba...",19.69,$/BBL


In [5]:
df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
df = df[['period', 'value', 'series']]
df_wide = df.pivot(index='period', columns = 'series', values= 'value')
df_wide = df_wide.dropna()
df_wide


series,RBRTE,RWTC
period,,
1987-05-15,18.58,19.52
1987-05-22,18.54,19.85
1987-05-29,18.60,19.34
1987-06-05,18.70,19.73
1987-06-12,18.75,19.88
...,...,...
2026-05-01,119.63,105.57
2026-05-08,105.88,102.28
2026-05-15,110.53,105.10


In [6]:
def fetch_eia_series(series_ids=['RBRTE', 'RWTC'], # ID's of series we're pulling (Brent, WTI)
                          frequency= 'weekly',# Frequency of the spot prices, either daily, weekly, or monthly.
                            length = 5000 # Timespan that we're pulling, max 5000 weeks
                            ):
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)


    URL_BASE = f"https://api.eia.gov/v2/petroleum/pri/spt/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}"
    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide
fetch_eia_series()

series,RBRTE,RWTC
period,,
1987-05-15,18.58,19.52
1987-05-22,18.54,19.85
1987-05-29,18.60,19.34
1987-06-05,18.70,19.73
1987-06-12,18.75,19.88
...,...,...
2026-05-01,119.63,105.57
2026-05-08,105.88,102.28
2026-05-15,110.53,105.10


## FRED Series (DXY, VIX, 10Y - 2Y Spread)

In [7]:
def fetch_fred_series(series_ids=['T10Y2Y','VIXCLS','DTWEXBGS'], #ID's of the series we want to pull, here 10Y-2Y spread, VIX, and DXY
                      frequency = 'W-FRI'  # How frequent we want the samples to be. (day -> "D", weekly, on friday -> "W-FRI", monthly -> "M", yearly -> "Y"
                      ):
    
    load_dotenv()
    FRED_API_KEY = os.getenv('FRED_API_KEY')
    fred = Fred(api_key= FRED_API_KEY)
    series_dict = {}
    for id in series_ids: # Create a dict with key as ID and values as the corresponding series
        series_dict[id] = fred.get_series(series_id=id)

    df = pd.DataFrame(series_dict) # Turn dictionary into a DF
    df.index = pd.to_datetime(df.index) # Convert index dtype to datetime
    df.index.name = 'period' # Convert index name to 'period' to match the EIA information

    df = df.resample(rule=frequency).last() # Resample to keep only the days at the end of the week

    df = df.dropna() # Drop any NaNs

    return df
fetch_fred_series()

,T10Y2Y,VIXCLS,DTWEXBGS
period,,,
2006-01-06,0.02,11.00,100.0241
2006-01-13,0.02,11.23,99.9675
2006-01-20,0.00,14.56,99.9017
2006-01-27,0.01,11.97,99.6433
2006-02-03,-0.05,12.96,100.1180
...,...,...,...
2026-05-08,0.48,17.19,118.0392
2026-05-15,0.50,18.43,119.2825
2026-05-22,0.43,16.70,119.2868


In [ ]:
def get_merged_df(eia_df=fetch_eia_series(), fred_df=fetch_fred_series()):
    df = pd.merge(left=eia_df, right=fred_df,left_index = True, right_index = True, how= 'inner')
    return df
get_merged_df().head()

,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS
period,,,,,
2006-01-06,NaN,NaN,NaN,NaN,NaN
2006-01-13,61.72,63.39,0.02,11.00,100.0241
2006-01-20,62.18,63.74,0.02,11.23,99.9675
2006-01-27,63.54,66.79,0.00,14.56,99.9017
2006-02-03,63.77,66.82,0.01,11.97,99.6433
...,...,...,...,...,...
2026-05-01,109.62,95.43,0.53,18.71,118.7294
2026-05-08,119.63,105.57,0.51,16.99,118.3926
2026-05-15,105.88,102.28,0.48,17.19,118.0392


In [17]:
def compute_returns(df=get_merged_df(), method= 'log',series_ids= ['RBRTE','RWTC']):
    df = df.copy()
    for id in series_ids:
        df[id + '_log_return'] = np.log(df[id]/ df[id].shift(1))
    return df.dropna()

toy = compute_returns()

    
    


In [16]:
toy.sort_values(by= ['RBRTE_log_return'], ascending= False).head()

,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,RBRTE_log_return,RWTC_log_return
period,,,,,,,
2020-05-08,23.57,23.46,0.53,27.98,122.5379,0.323825,0.400999
2009-01-09,45.25,44.46,1.68,42.82,98.8752,0.200204,0.047442
2020-05-22,33.94,33.10,0.49,28.16,122.3211,0.185982,0.226169
2020-04-10,22.53,24.41,0.50,41.67,122.1592,0.185255,0.118142
2020-05-01,17.05,15.71,0.44,37.19,123.1492,0.180095,1.554333


In [27]:
def fetch_eia_stock(series_ids=['WCESTUS1'], # ID's of series we're pulling (Week-end US Crude Inventory)
                          frequency= 'weekly',# Frequency of the spot prices, either daily, weekly, or monthly.
                            length = 5000 # Timespan that we're pulling, max 5000 weeks
                            ):
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)

    URL_BASE = f'https://api.eia.gov/v2/petroleum/stoc/wstk/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}'

    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide

fetch_eia_stock().head()

series,WCESTUS1
period,
1982-08-20,338764
1982-08-27,336138
1982-09-24,335586
1982-10-01,334786
1982-10-08,335260


In [30]:
def get_merged_df(eia_spot_df=fetch_eia_series(), eia_stock_df=fetch_eia_stock(),fred_df=fetch_fred_series()):
    df = pd.merge(left=eia_spot_df, right=fred_df,left_index = True, right_index = True, how= 'inner')
    df = pd.merge(left=df, right=eia_stock_df, left_index = True, right_index = True, how = 'inner')
    return df

get_merged_df().head()

,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1
period,,,,,,
2006-01-06,61.72,63.39,0.02,11.00,100.0241,302584
2006-01-13,62.18,63.74,0.02,11.23,99.9675,305325
2006-01-20,63.54,66.79,0.00,14.56,99.9017,303016
2006-01-27,63.77,66.82,0.01,11.97,99.6433,304935
2006-02-03,64.00,66.59,-0.05,12.96,100.1180,304523
